# Dynamic prompts — a system prompt built at call time
**by Kassouar Fatima 🇩🇿 — Module 3 project**

`@dynamic_prompt` turns the system prompt into a function of the request: it reads `user_language` from the run context and appends one instruction — answer only in that language. The agent is built **once** and invoked three times (Irish, Spanish, French): three prompts, no rebuild.

- **Run:** double-click `run_3_4.cmd` (opens all three `3_4` notebooks) → *Run → Run All Cells*.
- **Needs:** `GOOGLE_API_KEY` from the `.env` next to this notebook.
- **Model:** `gemini-3.5-flash-lite` — its own 500/day bucket, free while `gemini-3.1-flash-lite` is spent. **Model fix:** the course used `gpt-5-nano` → Gemini here, no extra key.

**Idea:** context is per-run configuration, state is the conversation — the prompt reads context, so the *caller* decides how the agent behaves.

**What I learned:**
- The decorated function returns the **whole** prompt string — base part and conditional part together.
- Context arrives at invoke time: `context=LanguageContext(user_language="Irish")`, declared once via `context_schema`.
- It is evaluated on **every** model call — switch the language mid-thread and the next call obeys.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

# Install only what is missing. Uses pip when available, falls back to uv
# (the local venv on this machine is created by uv and has no pip).
REQUIRED = {
    "langchain": "langchain",
    "langgraph": "langgraph",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_community": "langchain-community",
    "dotenv": "python-dotenv",
}
missing = [pkg for imp, pkg in REQUIRED.items() if _ilu.find_spec(imp) is None]

if missing:
    print("Installing:", ", ".join(missing))
    if _sp.run([_sys.executable, "-m", "pip", "install", "-q", *missing]).returncode != 0:
        _sp.run(["uv", "pip", "install", *missing])
else:
    print("All packages already installed.")

All packages already installed.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # local run: finds the .env above the notebook folder

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..." ) count as missing
    return bool(value) and not value.startswith("your_")

# Accept both names: the local project keeps the key as GEMINI_API_KEY
for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if is_real(os.getenv(name)):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = os.getenv(name)
        break

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("Gemini API key is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [3]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Dia duit! Tá mé go maith, go raibh maith agat. Conas atá tú féin?', 'extras': {'signature': 'EmAKXgFpFH0TPFBQsTR0cqzCyVtJSwBlDyjL9ifynH/pAdtaPaoPBYOCmhzYqgqeacOX5PEZp9rVevHk/rqg+BSD5/vuVtFy0IDx/Dwdvtea1bhgn4/Hqv6Kt0L1alaFCOo='}}]


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': '¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué te puedo ayudar hoy?', 'extras': {'signature': 'EmAKXgFpFH0TxA/HgtGr7UPKFHAZcihHSD7qss7RLY8i3T8rUH93XVolzZJipKpIbysOX9yPhmoaT6ZWOiZ5AlStPx82fwdznS0Mrr+cd1vVK1NKVqvZfvtDc8PPQeu71sU='}}]


In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Bonjour ! Je vais très bien, merci. Comment puis-je vous aider aujourd'hui ?", 'extras': {'signature': 'EmAKXgFpFH0T+u9dIlhnw5UoT8ojHINPcgvNgEeBHkwBYDEpwig3cVRSTJm3YfGxMyxhH80uJXXCs4EflmoDNrbRr/UVVqtay8A7TXeyu7vkzWbch+pcGJLl0O4LAhwNGVw='}}]


## Your task

> - Make one prompt dynamic. Anything: language, tone, level of detail. Get it switching on something real.
> - Hide a tool from one kind of user. This is the pattern behind every permission you will ever implement.
> - Then try the model switch, and put the token counts side by side. That is the difference between an agent that works and an agent that can afford to run.

### My response — by Kassouar Fatima 🇩🇿 — Module 3 project

This notebook does **task 1**; tasks 2 and 3 live in the two sibling notebooks
(`3_4_Dynamic_tools_kassouar`, `3_4_Dynamic_models_kassouar`).

**The dynamic prompt.** `@dynamic_prompt` turns the system prompt into a function of the request:

```python
@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."
    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt
```

It returns the **whole** prompt, not an add-on — the agent is built once with
`context_schema=LanguageContext` and the caller decides the language on every invoke.

**Switching on something real** — one agent, one identical message (*"Hello, how are you?"*), three
runs with `context=LanguageContext(user_language=...)`:

| `user_language` | what came back |
|---|---|
| `Irish` | **"Dia duit! Tá mé go maith, go raibh maith agat. Conas atá tú féin?"** |
| `Spanish` | **"¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué te puedo ayudar hoy?"** |
| `French` | **"Bonjour ! Je vais très bien, merci. Comment puis-je vous aider aujourd'hui ?"** |

`English` gets no suffix, so the base prompt is untouched. Same model, same tools, same message —
the only difference between those three answers is one field read at call time. That is the whole
pattern: the prompt stops being a constant baked in at build time and becomes data the caller
supplies.

## Summary — main functions & parameters

| Function / parameter | What it does |
|---|---|
| `@dataclass LanguageContext(user_language="English")` | the per-request setting; defaults to `English` so nothing breaks when the caller says nothing |
| `@dynamic_prompt user_language_prompt(request: ModelRequest) -> str` | returns the **whole** system prompt, built at call time |
| `request.runtime.context.user_language` | where the value comes from at invoke time |
| `f"{base_prompt} only respond in {user_language}."` | the conditional branch; `English` gets no suffix, so the base prompt is untouched |
| `create_agent(..., context_schema=LanguageContext, middleware=[user_language_prompt])` | declares which context fields exist and which prompt builder to run |
| `agent.invoke(..., context=LanguageContext(user_language="Irish"))` | the caller decides the language per call — one agent, three answers |

---
*by Kassouar Fatima 🇩🇿 — Module 3 project*